# Нечеткая система оценки качества сна

Лабораторная работа выполнена на основе исходного notebook-примера, но предметная область заменена на систему оценки качества сна. Для реализации используется библиотека `scikit-fuzzy`: модуль `skfuzzy` для функций принадлежности и дефаззификации, модуль `skfuzzy.control` для построения системы Мамдани.

## Входные переменные

1. Длительность сна `SleepDuration`, область `0-12` часов: очень короткий, короткий, нормальный, длительный.
2. Количество пробуждений `Awakenings`, область `0-10`: нет, мало, средне, много.
3. Уровень шума `NoiseLevel`, область `20-90` дБ: тихо, умеренно, шумно, очень шумно.
4. Уровень стресса `StressLevel`, область `0-10`: низкий, средний, высокий, критический.

## Выходная переменная

Качество сна `SleepQuality`, область `0-100`: критически плохое, плохое, удовлетворительное, хорошее.


In [1]:
from pathlib import Path
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

plt.rcParams["figure.figsize"] = (14, 9)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

FIGURES_DIR = Path("figures")
OUTPUT_DIR = Path("output")
FIGURES_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"scikit-fuzzy подключен: {fuzz.__name__}")


scikit-fuzzy подключен: skfuzzy


## 1. Создание лингвистических переменных и функций принадлежности


In [2]:
# Создание входных переменных
sleep_duration = ctrl.Antecedent(np.arange(0, 12.01, 0.01), "sleep_duration")
awakenings = ctrl.Antecedent(np.arange(0, 10.01, 0.01), "awakenings")
noise_level = ctrl.Antecedent(np.arange(20, 90.1, 0.1), "noise_level")
stress_level = ctrl.Antecedent(np.arange(0, 10.01, 0.01), "stress_level")

# Создание выходной переменной
sleep_quality = ctrl.Consequent(np.arange(0, 100.1, 0.1), "sleep_quality")

# Функции принадлежности для длительности сна
sleep_duration["very_short"] = fuzz.trimf(sleep_duration.universe, [0, 0, 4])
sleep_duration["short"] = fuzz.trimf(sleep_duration.universe, [3, 4.5, 6])
sleep_duration["normal"] = fuzz.trimf(sleep_duration.universe, [5.5, 7.25, 9])
sleep_duration["long"] = fuzz.trimf(sleep_duration.universe, [8, 12, 12])

# Функции принадлежности для количества пробуждений
awakenings["none"] = fuzz.trimf(awakenings.universe, [0, 0, 1])
awakenings["few"] = fuzz.trimf(awakenings.universe, [0, 1.5, 3])
awakenings["medium"] = fuzz.trimf(awakenings.universe, [2, 4, 6])
awakenings["many"] = fuzz.trimf(awakenings.universe, [5, 10, 10])

# Функции принадлежности для уровня шума
noise_level["quiet"] = fuzz.trimf(noise_level.universe, [20, 20, 35])
noise_level["moderate"] = fuzz.trimf(noise_level.universe, [30, 40, 50])
noise_level["noisy"] = fuzz.trimf(noise_level.universe, [45, 57.5, 70])
noise_level["very_noisy"] = fuzz.trimf(noise_level.universe, [65, 90, 90])

# Функции принадлежности для уровня стресса
stress_level["low"] = fuzz.trimf(stress_level.universe, [0, 0, 3])
stress_level["medium"] = fuzz.trimf(stress_level.universe, [2, 4, 6])
stress_level["high"] = fuzz.trimf(stress_level.universe, [5, 6.5, 8])
stress_level["critical"] = fuzz.trimf(stress_level.universe, [7, 10, 10])

# Функции принадлежности для качества сна
sleep_quality["critical"] = fuzz.trimf(sleep_quality.universe, [0, 0, 30])
sleep_quality["bad"] = fuzz.trimf(sleep_quality.universe, [20, 35, 50])
sleep_quality["satisfactory"] = fuzz.trimf(sleep_quality.universe, [45, 60, 75])
sleep_quality["good"] = fuzz.trimf(sleep_quality.universe, [70, 100, 100])

variables = {
    "sleep_duration": sleep_duration,
    "awakenings": awakenings,
    "noise_level": noise_level,
    "stress_level": stress_level,
    "sleep_quality": sleep_quality,
}

term_labels = {
    "very_short": "Очень короткий",
    "short": "Короткий",
    "normal": "Нормальный",
    "long": "Длительный",
    "none": "Нет",
    "few": "Мало",
    "medium": "Средне",
    "many": "Много",
    "quiet": "Тихо",
    "moderate": "Умеренно",
    "noisy": "Шумно",
    "very_noisy": "Очень шумно",
    "low": "Низкий",
    "high": "Высокий",
    "critical": "Критически плохое / критический",
    "bad": "Плохое",
    "satisfactory": "Удовлетворительное",
    "good": "Хорошее",
}

output_labels = {
    "critical": "Критически плохое",
    "bad": "Плохое",
    "satisfactory": "Удовлетворительное",
    "good": "Хорошее",
}

plot_terms = {
    "sleep_duration": [("very_short", "Очень короткий"), ("short", "Короткий"), ("normal", "Нормальный"), ("long", "Длительный")],
    "awakenings": [("none", "Нет"), ("few", "Мало"), ("medium", "Средне"), ("many", "Много")],
    "noise_level": [("quiet", "Тихо"), ("moderate", "Умеренно"), ("noisy", "Шумно"), ("very_noisy", "Очень шумно")],
    "stress_level": [("low", "Низкий"), ("medium", "Средний"), ("high", "Высокий"), ("critical", "Критический")],
    "sleep_quality": [("critical", "Критически плохое"), ("bad", "Плохое"), ("satisfactory", "Удовлетворительное"), ("good", "Хорошее")],
}

variable_titles = {
    "sleep_duration": ("Длительность сна", "Часы"),
    "awakenings": ("Количество пробуждений", "Раз за ночь"),
    "noise_level": ("Уровень шума", "дБ"),
    "stress_level": ("Уровень стресса", "Баллы"),
    "sleep_quality": ("Качество сна", "Условные единицы"),
}

colors = ["#d62728", "#ff7f0e", "#1f77b4", "#2ca02c"]

print("Лингвистические переменные и функции принадлежности созданы.")


Лингвистические переменные и функции принадлежности созданы.


In [3]:
def plot_variable(ax, variable_key):
    variable = variables[variable_key]
    title, xlabel = variable_titles[variable_key]
    for (term_key, label), color in zip(plot_terms[variable_key], colors):
        ax.plot(variable.universe, variable[term_key].mf, linewidth=2, color=color, label=label)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Степень принадлежности")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(3, 2, figsize=(14, 13))
for ax, variable_key in zip(axes.flat, ["sleep_duration", "awakenings", "noise_level", "stress_level", "sleep_quality"]):
    plot_variable(ax, variable_key)
axes.flat[-1].axis("off")
fig.suptitle("Функции принадлежности нечеткой системы оценки качества сна", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIGURES_DIR / "membership_functions.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\2066864100.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Создание базы знаний продукций


In [4]:
rule_specs = [
    (("normal", "none", "quiet", "low"), "good", "Длительность=Нормальный И Пробуждения=Нет И Шум=Тихо И Стресс=Низкий"),
    (("normal", "few", "quiet", "low"), "good", "Длительность=Нормальный И Пробуждения=Мало И Шум=Тихо И Стресс=Низкий"),
    (("normal", "none", "moderate", "low"), "good", "Длительность=Нормальный И Пробуждения=Нет И Шум=Умеренно И Стресс=Низкий"),
    (("long", "none", "quiet", "low"), "good", "Длительность=Длительный И Пробуждения=Нет И Шум=Тихо И Стресс=Низкий"),
    (("normal", "few", "moderate", "low"), "good", "Длительность=Нормальный И Пробуждения=Мало И Шум=Умеренно И Стресс=Низкий"),
    (("long", "few", "quiet", "low"), "good", "Длительность=Длительный И Пробуждения=Мало И Шум=Тихо И Стресс=Низкий"),
    (("normal", "none", "quiet", "medium"), "good", "Длительность=Нормальный И Пробуждения=Нет И Шум=Тихо И Стресс=Средний"),
    (("long", "none", "moderate", "low"), "good", "Длительность=Длительный И Пробуждения=Нет И Шум=Умеренно И Стресс=Низкий"),

    (("normal", "medium", "quiet", "low"), "satisfactory", "Длительность=Нормальный И Пробуждения=Средне И Шум=Тихо И Стресс=Низкий"),
    (("normal", "few", "noisy", "low"), "satisfactory", "Длительность=Нормальный И Пробуждения=Мало И Шум=Шумно И Стресс=Низкий"),
    (("normal", "few", "moderate", "medium"), "satisfactory", "Длительность=Нормальный И Пробуждения=Мало И Шум=Умеренно И Стресс=Средний"),
    (("long", "medium", "quiet", "low"), "satisfactory", "Длительность=Длительный И Пробуждения=Средне И Шум=Тихо И Стресс=Низкий"),
    (("short", "none", "quiet", "low"), "satisfactory", "Длительность=Короткий И Пробуждения=Нет И Шум=Тихо И Стресс=Низкий"),
    (("short", "few", "quiet", "low"), "satisfactory", "Длительность=Короткий И Пробуждения=Мало И Шум=Тихо И Стресс=Низкий"),
    (("normal", "none", "moderate", "medium"), "satisfactory", "Длительность=Нормальный И Пробуждения=Нет И Шум=Умеренно И Стресс=Средний"),
    (("long", "few", "moderate", "medium"), "satisfactory", "Длительность=Длительный И Пробуждения=Мало И Шум=Умеренно И Стресс=Средний"),
    (("normal", "medium", "moderate", "medium"), "satisfactory", "Длительность=Нормальный И Пробуждения=Средне И Шум=Умеренно И Стресс=Средний"),
    (("long", "none", "noisy", "medium"), "satisfactory", "Длительность=Длительный И Пробуждения=Нет И Шум=Шумно И Стресс=Средний"),

    (("short", "medium", "moderate", "medium"), "bad", "Длительность=Короткий И Пробуждения=Средне И Шум=Умеренно И Стресс=Средний"),
    (("short", "many", "quiet", "medium"), "bad", "Длительность=Короткий И Пробуждения=Много И Шум=Тихо И Стресс=Средний"),
    (("short", "medium", "noisy", "medium"), "bad", "Длительность=Короткий И Пробуждения=Средне И Шум=Шумно И Стресс=Средний"),
    (("normal", "many", "moderate", "medium"), "bad", "Длительность=Нормальный И Пробуждения=Много И Шум=Умеренно И Стресс=Средний"),
    (("normal", "medium", "noisy", "high"), "bad", "Длительность=Нормальный И Пробуждения=Средне И Шум=Шумно И Стресс=Высокий"),
    (("long", "many", "moderate", "high"), "bad", "Длительность=Длительный И Пробуждения=Много И Шум=Умеренно И Стресс=Высокий"),
    (("short", "few", "very_noisy", "medium"), "bad", "Длительность=Короткий И Пробуждения=Мало И Шум=Очень шумно И Стресс=Средний"),
    (("short", "medium", "moderate", "high"), "bad", "Длительность=Короткий И Пробуждения=Средне И Шум=Умеренно И Стресс=Высокий"),
    (("normal", "many", "noisy", "medium"), "bad", "Длительность=Нормальный И Пробуждения=Много И Шум=Шумно И Стресс=Средний"),
    (("long", "medium", "noisy", "high"), "bad", "Длительность=Длительный И Пробуждения=Средне И Шум=Шумно И Стресс=Высокий"),

    (("very_short", "many", "very_noisy", "critical"), "critical", "Длительность=Очень короткий И Пробуждения=Много И Шум=Очень шумно И Стресс=Критический"),
    (("very_short", "medium", "noisy", "high"), "critical", "Длительность=Очень короткий И Пробуждения=Средне И Шум=Шумно И Стресс=Высокий"),
    (("very_short", "many", "noisy", "high"), "critical", "Длительность=Очень короткий И Пробуждения=Много И Шум=Шумно И Стресс=Высокий"),
    (("very_short", "many", "moderate", "critical"), "critical", "Длительность=Очень короткий И Пробуждения=Много И Шум=Умеренно И Стресс=Критический"),
    (("short", "many", "very_noisy", "high"), "critical", "Длительность=Короткий И Пробуждения=Много И Шум=Очень шумно И Стресс=Высокий"),
    (("short", "many", "noisy", "critical"), "critical", "Длительность=Короткий И Пробуждения=Много И Шум=Шумно И Стресс=Критический"),
    (("very_short", "none", "quiet", "critical"), "bad", "Длительность=Очень короткий И Пробуждения=Нет И Шум=Тихо И Стресс=Критический"),
    (("very_short", "few", "moderate", "high"), "bad", "Длительность=Очень короткий И Пробуждения=Мало И Шум=Умеренно И Стресс=Высокий"),
    (("normal", "many", "very_noisy", "critical"), "critical", "Длительность=Нормальный И Пробуждения=Много И Шум=Очень шумно И Стресс=Критический"),
    (("long", "many", "very_noisy", "critical"), "critical", "Длительность=Длительный И Пробуждения=Много И Шум=Очень шумно И Стресс=Критический"),
    (("short", "medium", "very_noisy", "critical"), "critical", "Длительность=Короткий И Пробуждения=Средне И Шум=Очень шумно И Стресс=Критический"),
    (("very_short", "medium", "very_noisy", "critical"), "critical", "Длительность=Очень короткий И Пробуждения=Средне И Шум=Очень шумно И Стресс=Критический"),
]

def make_rule(index, antecedent_terms, consequent_term):
    sd, aw, nl, sl = antecedent_terms
    condition = (
        sleep_duration[sd]
        & awakenings[aw]
        & noise_level[nl]
        & stress_level[sl]
    )
    return ctrl.Rule(condition, sleep_quality[consequent_term], label=f"R{index}")

rules = [
    make_rule(index, antecedents, consequent)
    for index, (antecedents, consequent, _description) in enumerate(rule_specs, start=1)
]

sleep_control = ctrl.ControlSystem(rules)
sleep_simulation = ctrl.ControlSystemSimulation(sleep_control)

print("База знаний успешно создана.")
print(f"Количество правил: {len(rules)}")


База знаний успешно создана.
Количество правил: 40


## 3. Фаззификация контрольного примера


In [5]:
test_input = {
    "sleep_duration": 5.8,
    "awakenings": 2.5,
    "noise_level": 47.0,
    "stress_level": 5.5,
}

input_order = ["sleep_duration", "awakenings", "noise_level", "stress_level"]
input_units = {
    "sleep_duration": "ч",
    "awakenings": "раза",
    "noise_level": "дБ",
    "stress_level": "баллов",
}


def term_label(variable_key, term_key):
    return dict(plot_terms[variable_key])[term_key]


def membership_degrees(variable_key, value):
    variable = variables[variable_key]
    return {
        term_key: float(fuzz.interp_membership(variable.universe, variable[term_key].mf, value))
        for term_key, _label in plot_terms[variable_key]
    }


def membership_bar(value, width=50):
    return "█" * int(round(value * width))


degrees = {
    variable_key: membership_degrees(variable_key, value)
    for variable_key, value in test_input.items()
}

print("\n" + "=" * 70)
print("ПРИМЕР ФАЗЗИФИКАЦИИ")
print("=" * 70)

print("\nВходные параметры для оценки качества сна:")
for variable_key in input_order:
    title, _xlabel = variable_titles[variable_key]
    print(f"  {title}: {test_input[variable_key]:g} {input_units[variable_key]}")

for variable_key in input_order:
    title, _xlabel = variable_titles[variable_key]
    print(f"\n{'─' * 70}")
    print(f"Степени принадлежности для {title} = {test_input[variable_key]:g} {input_units[variable_key]}:")
    print(f"{'─' * 70}")
    for term_key, label in plot_terms[variable_key]:
        degree = degrees[variable_key][term_key]
        print(f"  {label:<18} {degree:.4f} {membership_bar(degree)}")

print("\n" + "=" * 70)



ПРИМЕР ФАЗЗИФИКАЦИИ

Входные параметры для оценки качества сна:
  Длительность сна: 5.8 ч
  Количество пробуждений: 2.5 раза
  Уровень шума: 47 дБ
  Уровень стресса: 5.5 баллов

──────────────────────────────────────────────────────────────────────
Степени принадлежности для Длительность сна = 5.8 ч:
──────────────────────────────────────────────────────────────────────
  Очень короткий     0.0000 
  Короткий           0.1333 ███████
  Нормальный         0.1714 █████████
  Длительный         0.0000 

──────────────────────────────────────────────────────────────────────
Степени принадлежности для Количество пробуждений = 2.5 раза:
──────────────────────────────────────────────────────────────────────
  Нет                0.0000 
  Мало               0.3333 █████████████████
  Средне             0.2500 ████████████
  Много              0.0000 

──────────────────────────────────────────────────────────────────────
Степени принадлежности для Уровень шума = 47 дБ:
───────────────────────

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, variable_key in zip(axes.flat, input_order):
    plot_variable(ax, variable_key)
    value = test_input[variable_key]
    ax.axvline(value, color="black", linestyle="--", linewidth=2.5, label=f"Значение: {value:g}")
    y_values = [degrees[variable_key][term_key] for term_key, _label in plot_terms[variable_key]]
    ax.scatter(
        [value] * len(y_values),
        y_values,
        color="black",
        s=95,
        zorder=5,
        marker="o",
        edgecolors="white",
        linewidths=1.8,
    )
    title, xlabel = variable_titles[variable_key]
    ax.set_title(f"Фаззификация: {title} = {value:g} {input_units[variable_key]}", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8, loc="upper right")

fig.suptitle("Фаззификация контрольного примера", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES_DIR / "fuzzification_example.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\359677484.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Активизация и агрегирование правил


In [7]:
print("=" * 80)
print("4. АКТИВИЗАЦИЯ И АГРЕГИРОВАНИЕ ПРАВИЛ НЕЧЕТКИХ ПРОДУКЦИЙ")
print("=" * 80)

print("\nВходные параметры для оценки качества сна:")
for variable_key in input_order:
    title, _xlabel = variable_titles[variable_key]
    print(f"  {title}: {test_input[variable_key]:g} {input_units[variable_key]}")

print("\n" + "-" * 80)
print("ЭТАП 1: ФАЗЗИФИКАЦИЯ")
print("-" * 80)
for variable_key in input_order:
    title, _xlabel = variable_titles[variable_key]
    print(f"\n{title} = {test_input[variable_key]:g} {input_units[variable_key]}:")
    for term_key, label in plot_terms[variable_key]:
        print(f"  μ({label}) = {degrees[variable_key][term_key]:.4f}")

# Расчет результата через scikit-fuzzy ControlSystemSimulation
sleep_simulation = ctrl.ControlSystemSimulation(sleep_control)
for variable_key, value in test_input.items():
    sleep_simulation.input[variable_key] = value
sleep_simulation.compute()
output_value = float(sleep_simulation.output["sleep_quality"])


def activate_rule(antecedents):
    return min(
        degrees["sleep_duration"][antecedents[0]],
        degrees["awakenings"][antecedents[1]],
        degrees["noise_level"][antecedents[2]],
        degrees["stress_level"][antecedents[3]],
    )


def rule_degree_parts(antecedents):
    return [degrees[variable_key][term_key] for variable_key, term_key in zip(input_order, antecedents)]


evaluated_rules = []
consequent_levels = {term_key: 0.0 for term_key, _label in plot_terms["sleep_quality"]}
for index, (antecedents, consequent, description) in enumerate(rule_specs, start=1):
    activation = activate_rule(antecedents)
    consequent_levels[consequent] = max(consequent_levels[consequent], activation)
    evaluated_rules.append(
        {
            "number": index,
            "antecedents": antecedents,
            "consequent": consequent,
            "description": description,
            "activation": activation,
            "parts": rule_degree_parts(antecedents),
        }
    )

active_rules = [rule for rule in evaluated_rules if rule["activation"] > 1e-9]
active_sorted = sorted(active_rules, key=lambda rule: rule["activation"], reverse=True)
top_active_rules = active_sorted[:2]

print("\n" + "-" * 80)
print("ЭТАП 2: АКТИВИЗАЦИЯ ПРАВИЛ (метод min)")
print("-" * 80)

if active_sorted:
    for rule in active_sorted:
        parts = ", ".join(f"{value:.4f}" for value in rule["parts"])
        print(f"\nПравило {rule['number']}: {rule['description']}")
        print(f"  ТО Качество сна = {output_labels[rule['consequent']]}")
        print(f"  Степень активизации: min({parts}) = {rule['activation']:.4f}")
else:
    print("\nАктивных правил для заданного контрольного примера не найдено.")

aggregated = np.zeros_like(sleep_quality.universe)
activated_output_terms = {}
for term_key, _label in plot_terms["sleep_quality"]:
    clipped = np.fmin(consequent_levels[term_key], sleep_quality[term_key].mf)
    activated_output_terms[term_key] = clipped
    aggregated = np.fmax(aggregated, clipped)

print("\n" + "-" * 80)
print("ЭТАП 3: АГРЕГИРОВАНИЕ (метод max)")
print("-" * 80)
print("\nИтоговые уровни активации выходных термов:")
for term_key, value in consequent_levels.items():
    print(f"  {output_labels[term_key]:22s} = {value:.4f}")
print("\nАгрегирование всех активированных правил выполняется методом максимума (OR).")
print(f"Результирующее значение качества сна: {output_value:.2f}")


4. АКТИВИЗАЦИЯ И АГРЕГИРОВАНИЕ ПРАВИЛ НЕЧЕТКИХ ПРОДУКЦИЙ

Входные параметры для оценки качества сна:
  Длительность сна: 5.8 ч
  Количество пробуждений: 2.5 раза
  Уровень шума: 47 дБ
  Уровень стресса: 5.5 баллов

--------------------------------------------------------------------------------
ЭТАП 1: ФАЗЗИФИКАЦИЯ
--------------------------------------------------------------------------------

Длительность сна = 5.8 ч:
  μ(Очень короткий) = 0.0000
  μ(Короткий) = 0.1333
  μ(Нормальный) = 0.1714
  μ(Длительный) = 0.0000

Количество пробуждений = 2.5 раза:
  μ(Нет) = 0.0000
  μ(Мало) = 0.3333
  μ(Средне) = 0.2500
  μ(Много) = 0.0000

Уровень шума = 47 дБ:
  μ(Тихо) = 0.0000
  μ(Умеренно) = 0.3000
  μ(Шумно) = 0.1600
  μ(Очень шумно) = 0.0000

Уровень стресса = 5.5 баллов:
  μ(Низкий) = 0.0000
  μ(Средний) = 0.2500
  μ(Высокий) = 0.3333
  μ(Критический) = 0.0000

--------------------------------------------------------------------------------
ЭТАП 2: АКТИВИЗАЦИЯ ПРАВИЛ (метод min)
-----

In [8]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
flat_axes = axes.flat

# Графики 1-4: входные переменные с текущими значениями
for ax, variable_key in zip(flat_axes[:4], input_order):
    plot_variable(ax, variable_key)
    value = test_input[variable_key]
    ax.axvline(value, color="black", linewidth=2.4, linestyle="--", label=f"Значение: {value:g}")
    y_values = [degrees[variable_key][term_key] for term_key, _label in plot_terms[variable_key]]
    ax.scatter([value] * len(y_values), y_values, color="black", s=55, zorder=5, edgecolors="white", linewidths=1.2)
    ax.set_title(f"{variable_titles[variable_key][0]} с текущим значением", fontweight="bold", fontsize=11)
    ax.set_ylabel("μ")
    ax.legend(fontsize=7, loc="upper right")

# Графики 5-6: активизация ключевых правил
for ax, rule in zip(flat_axes[4:6], top_active_rules):
    term_key = rule["consequent"]
    activation = rule["activation"]
    activated = np.fmin(activation, sleep_quality[term_key].mf)
    color = colors[[term for term, _label in plot_terms["sleep_quality"]].index(term_key)]
    ax.fill_between(sleep_quality.universe, 0, activated, alpha=0.7, color=color)
    ax.plot(sleep_quality.universe, sleep_quality[term_key].mf, color=color, linewidth=2, alpha=0.35)
    ax.set_title(f"Активизация R{rule['number']}: {output_labels[term_key]}\nμ = {activation:.4f}", fontweight="bold", fontsize=11)
    ax.set_xlabel("Качество сна")
    ax.set_ylabel("μ")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

for ax in flat_axes[4 + len(top_active_rules):6]:
    ax.axis("off")

# График 7: выходные функции принадлежности
ax = flat_axes[6]
for (term_key, label), color in zip(plot_terms["sleep_quality"], colors):
    ax.plot(sleep_quality.universe, sleep_quality[term_key].mf, color=color, linestyle="--", linewidth=1.8, alpha=0.65, label=label)
ax.set_title("Выходные функции принадлежности", fontweight="bold", fontsize=11)
ax.set_xlabel("Качество сна")
ax.set_ylabel("μ")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# График 8: агрегирование
ax = flat_axes[7]
ax.fill_between(sleep_quality.universe, 0, aggregated, alpha=0.65, color="#7b3294")
ax.plot(sleep_quality.universe, aggregated, color="#7b3294", linewidth=2.5, label="Агрегированная функция")
ax.set_title("Агрегирование (MAX)", fontweight="bold", fontsize=11)
ax.set_xlabel("Качество сна")
ax.set_ylabel("μ")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# График 9: результат дефаззификации
ax = flat_axes[8]
ax.fill_between(sleep_quality.universe, 0, aggregated, alpha=0.65, color="#7b3294")
ax.plot(sleep_quality.universe, aggregated, color="#7b3294", linewidth=2.5)
ax.axvline(output_value, color="#d62728", linewidth=3, linestyle="--", label=f"Центроид: {output_value:.2f}")
ax.set_title("Результат дефаззификации", fontweight="bold", fontsize=11)
ax.set_xlabel("Качество сна")
ax.set_ylabel("μ")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle("Активизация и агрегирование правил нечеткого вывода", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES_DIR / "activation_aggregation.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\2177106095.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
print(f"\n{'=' * 80}")
print("РЕЗУЛЬТАТ")
print(f"{'=' * 80}")
print(f"\nИтоговая оценка качества сна: {output_value:.2f}")

if output_value <= 30:
    result_interpretation = "КРИТИЧЕСКИ ПЛОХОЕ КАЧЕСТВО СНА - выраженные неблагоприятные условия и высокий риск недосыпа"
elif output_value <= 50:
    result_interpretation = "ПЛОХОЕ КАЧЕСТВО СНА - требуется улучшить условия сна и снизить стрессовую нагрузку"
elif output_value <= 75:
    result_interpretation = "УДОВЛЕТВОРИТЕЛЬНОЕ КАЧЕСТВО СНА - состояние приемлемое, но есть факторы, ухудшающие восстановление"
else:
    result_interpretation = "ХОРОШЕЕ КАЧЕСТВО СНА - параметры близки к благоприятным"

print(f"Интерпретация: {result_interpretation}")
print(f"\n{'=' * 80}")



РЕЗУЛЬТАТ

Итоговая оценка качества сна: 47.91
Интерпретация: ПЛОХОЕ КАЧЕСТВО СНА - требуется улучшить условия сна и снизить стрессовую нагрузку



## 5. Дефаззификация различными методами


In [10]:
def interpretation(value):
    if value <= 30:
        return "критически плохое качество сна"
    if value <= 50:
        return "плохое качество сна"
    if value <= 75:
        return "удовлетворительное качество сна"
    return "хорошее качество сна"

print("\n" + "=" * 80)
print("5. ДЕФАЗЗИФИКАЦИЯ РАЗЛИЧНЫМИ МЕТОДАМИ")
print("=" * 80)

print("\nВходные параметры для оценки качества сна:")
for variable_key in input_order:
    title, _xlabel = variable_titles[variable_key]
    print(f"  {title}: {test_input[variable_key]:g} {input_units[variable_key]}")

defuzzification = {
    "centroid": float(fuzz.defuzz(sleep_quality.universe, aggregated, "centroid")),
    "bisector": float(fuzz.defuzz(sleep_quality.universe, aggregated, "bisector")),
    "mom": float(fuzz.defuzz(sleep_quality.universe, aggregated, "mom")),
    "som": float(fuzz.defuzz(sleep_quality.universe, aggregated, "som")),
    "lom": float(fuzz.defuzz(sleep_quality.universe, aggregated, "lom")),
}
defuzzification["coa"] = defuzzification["centroid"]

print("\n1. ЦЕНТР ТЯЖЕСТИ (Centroid / Center of Area - COA):")
print("   Формула: x_COG = Σ(μ(x) * x) / Σ(μ(x))")
print(f"   Результат: {defuzzification['centroid']:.4f}")

print("\n2. БИССЕКТОР (Bisector):")
print("   Описание: делит площадь под агрегированной функцией на две равные части")
print(f"   Результат: {defuzzification['bisector']:.4f}")

print("\n3. СРЕДНЕЕ МАКСИМУМОВ (Mean of Maximum - MOM):")
print("   Описание: среднее значение всех точек с максимальной степенью принадлежности")
print(f"   Результат: {defuzzification['mom']:.4f}")

print("\n4. ЛЕВОЕ МОДАЛЬНОЕ ЗНАЧЕНИЕ (Smallest of Maximum - SOM):")
print("   Описание: минимальное значение из точек с максимальной степенью принадлежности")
print(f"   Результат: {defuzzification['som']:.4f}")

print("\n5. ПРАВОЕ МОДАЛЬНОЕ ЗНАЧЕНИЕ (Largest of Maximum - LOM):")
print("   Описание: максимальное значение из точек с максимальной степенью принадлежности")
print(f"   Результат: {defuzzification['lom']:.4f}")

print("\n6. ЦЕНТР ПЛОЩАДИ (COA):")
print("   Для данной реализации совпадает с центром тяжести")
print(f"   Результат: {defuzzification['coa']:.4f}")

print(f"\nИтоговая интерпретация по центру тяжести: {interpretation(defuzzification['centroid'])}.")



5. ДЕФАЗЗИФИКАЦИЯ РАЗЛИЧНЫМИ МЕТОДАМИ

Входные параметры для оценки качества сна:
  Длительность сна: 5.8 ч
  Количество пробуждений: 2.5 раза
  Уровень шума: 47 дБ
  Уровень стресса: 5.5 баллов

1. ЦЕНТР ТЯЖЕСТИ (Centroid / Center of Area - COA):
   Формула: x_COG = Σ(μ(x) * x) / Σ(μ(x))
   Результат: 47.9107

2. БИССЕКТОР (Bisector):
   Описание: делит площадь под агрегированной функцией на две равные части
   Результат: 48.3333

3. СРЕДНЕЕ МАКСИМУМОВ (Mean of Maximum - MOM):
   Описание: среднее значение всех точек с максимальной степенью принадлежности
   Результат: 60.0000

4. ЛЕВОЕ МОДАЛЬНОЕ ЗНАЧЕНИЕ (Smallest of Maximum - SOM):
   Описание: минимальное значение из точек с максимальной степенью принадлежности
   Результат: 47.6000

5. ПРАВОЕ МОДАЛЬНОЕ ЗНАЧЕНИЕ (Largest of Maximum - LOM):
   Описание: максимальное значение из точек с максимальной степенью принадлежности
   Результат: 72.4000

6. ЦЕНТР ПЛОЩАДИ (COA):
   Для данной реализации совпадает с центром тяжести
   Результа

In [11]:
method_specs = [
    ("centroid", "Центр тяжести", "#d62728"),
    ("bisector", "Биссектор площади", "#1f77b4"),
    ("mom", "Среднее максимумов", "#2ca02c"),
    ("som", "Левое модальное", "#ff7f0e"),
    ("lom", "Правое модальное", "#9467bd"),
    ("coa", "Центр площади (COA)", "#8c564b"),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, (method_key, title, color) in zip(axes.flat, method_specs):
    value = defuzzification[method_key]
    ax.fill_between(sleep_quality.universe, 0, aggregated, alpha=0.28, color="gray")
    ax.plot(sleep_quality.universe, aggregated, color="black", linewidth=2)
    ax.axvline(value, color=color, linestyle="--", linewidth=2.4, label=f"{value:.2f}")
    ax.scatter(value, np.interp(value, sleep_quality.universe, aggregated), color=color, edgecolor="black", s=60, zorder=5)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Качество сна")
    ax.set_ylabel("Степень принадлежности")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("Дефаззификация агрегированной функции разными методами", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIGURES_DIR / "defuzzification_methods.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\2177530203.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Дополнительные рисунки для отчета

Ниже формируются дополнительные рисунки, соответствующие структуре отчета.

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, variable_key in zip(axes.flat, ["sleep_duration", "awakenings", "noise_level", "stress_level"]):
    plot_variable(ax, variable_key)
fig.suptitle("Функции принадлежности входных переменных", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES_DIR / "membership_input_variables.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\3243607864.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_variable(ax, "sleep_quality")
fig.suptitle("Функции принадлежности выходной переменной", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(FIGURES_DIR / "membership_output_variable.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\2193608905.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# Скриншотоподобный вывод фаззификации для отчета
fuzz_lines = [
    "ВХОДНЫЕ ЗНАЧЕНИЯ",
    f"Длительность сна:       {test_input['sleep_duration']} ч",
    f"Количество пробуждений: {test_input['awakenings']} раза",
    f"Уровень шума:           {test_input['noise_level']} дБ",
    f"Уровень стресса:        {test_input['stress_level']} баллов",
    "",
    "СТЕПЕНИ ПРИНАДЛЕЖНОСТИ",
]
for variable_key in ["sleep_duration", "awakenings", "noise_level", "stress_level"]:
    title, _unit = variable_titles[variable_key]
    fuzz_lines.append("")
    fuzz_lines.append(f"{title}:")
    for term_key, label in plot_terms[variable_key]:
        degree = degrees[variable_key][term_key]
        bar = "█" * int(round(degree * 32))
        fuzz_lines.append(f"  {label:<18} μ = {degree:0.4f} {bar}")

fig, ax = plt.subplots(figsize=(12, 7.2))
ax.set_facecolor("#fbfbfb")
ax.axis("off")
ax.text(
    0.035,
    0.965,
    "\n".join(fuzz_lines),
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontfamily="DejaVu Sans Mono",
    fontsize=11,
    color="#111111",
)
ax.add_patch(
    plt.Rectangle(
        (0.015, 0.015),
        0.97,
        0.97,
        transform=ax.transAxes,
        fill=False,
        edgecolor="#b0b0b0",
        linewidth=1.2,
    )
)
fig.tight_layout(pad=0.5)
fig.savefig(FIGURES_DIR / "fuzzification_output.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\2165216586.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
active_sorted = sorted(active_rules, key=lambda rule: rule["activation"], reverse=True)
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, rule in zip(axes.flat, active_sorted):
    term_key = rule["consequent"]
    activation = rule["activation"]
    activated = np.fmin(activation, sleep_quality[term_key].mf)
    ax.fill_between(sleep_quality.universe, 0, activated, alpha=0.55, color="#4c78a8")
    ax.plot(sleep_quality.universe, sleep_quality[term_key].mf, color="#1f77b4", linewidth=2, alpha=0.55)
    ax.set_title(f"R{rule['number']} -> {output_labels[term_key]}\nμ = {activation:.4f}", fontweight="bold")
    ax.set_xlabel("Качество сна")
    ax.set_ylabel("Степень принадлежности")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
for ax in axes.flat[len(active_sorted):]:
    ax.axis("off")
fig.suptitle("Примеры активизации ключевых правил", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(FIGURES_DIR / "key_rule_activations.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\4158130790.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
fig, ax = plt.subplots(figsize=(11, 6))
for (term_key, label), color in zip(plot_terms["sleep_quality"], colors):
    ax.plot(sleep_quality.universe, sleep_quality[term_key].mf, color=color, linestyle="--", linewidth=1.8, alpha=0.55, label=label)
    if consequent_levels[term_key] > 0:
        ax.fill_between(sleep_quality.universe, 0, activated_output_terms[term_key], color=color, alpha=0.25)
ax.plot(sleep_quality.universe, aggregated, color="black", linewidth=2.4, label="Агрегированная функция")
ax.axvline(defuzzification["centroid"], color="#d62728", linestyle="--", linewidth=2.2, label=f"Центр тяжести: {defuzzification['centroid']:.2f}")
ax.set_title("Агрегирование всех активированных правил методом максимума", fontweight="bold")
ax.set_xlabel("Качество сна")
ax.set_ylabel("Степень принадлежности")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "aggregation_all_rules.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\40172171.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Скриншотоподобный вывод дефаззификации для отчета
defuzz_lines = [
    "ДЕФАЗЗИФИКАЦИЯ АГРЕГИРОВАННОЙ ФУНКЦИИ",
    "",
    f"Центр тяжести (Centroid / COA):     {defuzzification['centroid']:.4f}",
    f"Биссектор площади:                  {defuzzification['bisector']:.4f}",
    f"Среднее максимумов (MOM):           {defuzzification['mom']:.4f}",
    f"Левое модальное значение (SOM):     {defuzzification['som']:.4f}",
    f"Правое модальное значение (LOM):    {defuzzification['lom']:.4f}",
    f"Центр площади фигуры (COA):         {defuzzification['coa']:.4f}",
    "",
    f"Итоговая оценка качества сна:       {defuzzification['centroid']:.2f}",
    f"Интерпретация:                      {interpretation(defuzzification['centroid'])}",
]

fig, ax = plt.subplots(figsize=(11.5, 4.8))
ax.set_facecolor("#fbfbfb")
ax.axis("off")
ax.text(
    0.04,
    0.93,
    "\n".join(defuzz_lines),
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontfamily="DejaVu Sans Mono",
    fontsize=12,
    color="#111111",
)
ax.add_patch(
    plt.Rectangle(
        (0.015, 0.015),
        0.97,
        0.97,
        transform=ax.transAxes,
        fill=False,
        edgecolor="#b0b0b0",
        linewidth=1.2,
    )
)
fig.tight_layout(pad=0.5)
fig.savefig(FIGURES_DIR / "defuzzification_output.png", dpi=220, bbox_inches="tight")
plt.show()


C:\Users\ilchu\AppData\Local\Temp\ipykernel_22196\1797352163.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Заключение

Разработана нечеткая система оценки качества сна с четырьмя входными переменными, одной выходной переменной и базой знаний из 40 продукционных правил. Notebook повторяет структуру исходного примера: строит функции принадлежности, показывает фаззификацию контрольного примера, активизацию правил методом `min`, агрегирование выходных множеств методом `max` и дефаззификацию несколькими методами.

Для контрольного примера с длительностью сна `5.8` ч, `2.5` пробуждениями, шумом `47` дБ и стрессом `5.5` баллов итоговая оценка по методу центра тяжести попадает в область плохого качества сна. Это означает, что выбранная нечеткая система рекомендует улучшить условия сна и снизить стрессовую нагрузку.
